# Silco Jupyter Smoke Test

This notebook defines Pydantic models locally, maps them into Silco diagrams, and then runs an advanced flow simulation. The architecture notation comes from the real Silco diagram backend; the animation focuses on traffic and flow metrics instead of redrawing fake symbols.

In [ ]:
from collections import defaultdict, deque
from math import ceil, sin, pi
from pathlib import Path
import subprocess
import tempfile
from typing import Literal
import re
from xml.etree import ElementTree as ET

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, Image as IPyImage, SVG, display
from PIL import Image as PILImage, ImageChops
from pydantic import BaseModel, Field

from silco import diagram, kernel

plt.rcParams['figure.dpi'] = 130

discovered = kernel.discover()
print('Discovered plugins:', [(p.category, p.name) for p in discovered])
print('Renderers:', kernel.names('renderers'))
print('Styles:', kernel.names('styles'))

## Define Notebook-Local Pydantic Models

In [ ]:
NodeKind = Literal['actor', 'service', 'database', 'queue', 'cache', 'storage', 'external', 'component']


class ServiceModel(BaseModel):
    id: str
    label: str
    kind: NodeKind = 'service'
    group: str | None = None
    technology: str | None = None
    description: str | None = None
    line: int | None = None
    monthly_base_cost: float = 0.0
    cost_per_request: float = 0.0
    replicas: int = 1
    max_replicas: int = 6
    capacity_rps_per_replica: float = 100.0
    base_latency_ms: float = 20.0
    failure_rate: float = 0.0
    retry_multiplier: float = 0.0
    queue_decay: float = 0.0


class LinkModel(BaseModel):
    source: str
    target: str
    protocol: str | None = None
    label: str | None = None
    mode: Literal['connect', 'flow'] = 'connect'
    calls_per_request: float = 1.0
    latency_ms: float = 0.0
    loss_rate: float = 0.0


class ArchitectureModel(BaseModel):
    title: str
    direction: Literal['LR', 'RL', 'TB', 'BT'] = 'LR'
    groups: dict[str, str] = Field(default_factory=dict)
    services: list[ServiceModel]
    links: list[LinkModel]


class TrafficScenario(BaseModel):
    name: str
    entry_requests_per_second: dict[str, float]
    duration_steps: int = 24
    step_seconds: int = 300
    diurnal_amplitude: float = 0.25
    spike_step: int | None = None
    spike_multiplier: float = 1.0


## Create the Domain Model in the Notebook

In [ ]:
commerce = ArchitectureModel(
    title='Global Commerce Platform',
    direction='LR',
    groups={
        'edge': 'Edge',
        'app': 'Application',
        'data': 'Data Plane',
    },
    services=[
        ServiceModel(id='user', label='Customer', kind='actor', line=0),
        ServiceModel(id='web', label='Web App', kind='service', group='edge', technology='Next.js', line=0, monthly_base_cost=45, capacity_rps_per_replica=70, base_latency_ms=22, failure_rate=0.01, retry_multiplier=0.01),
        ServiceModel(id='api', label='API Gateway', kind='service', group='edge', technology='Envoy', line=1, monthly_base_cost=55, capacity_rps_per_replica=90, base_latency_ms=12, failure_rate=0.005, retry_multiplier=0.02),
        ServiceModel(id='checkout', label='Checkout Service', kind='service', group='app', technology='Python', line=1, monthly_base_cost=80, capacity_rps_per_replica=60, base_latency_ms=45, failure_rate=0.02, retry_multiplier=0.04, cost_per_request=0.0015),
        ServiceModel(id='payments', label='Payments Provider', kind='external', line=1, base_latency_ms=90, failure_rate=0.03, retry_multiplier=0.08, cost_per_request=0.025),
        ServiceModel(id='queue', label='Order Events', kind='queue', group='data', technology='Kafka', line=2, monthly_base_cost=65, capacity_rps_per_replica=150, base_latency_ms=15, queue_decay=0.65, cost_per_request=0.0002),
        ServiceModel(id='orders', label='Orders DB', kind='database', group='data', technology='Postgres', line=2, monthly_base_cost=120, capacity_rps_per_replica=80, base_latency_ms=30, failure_rate=0.01, cost_per_request=0.0004),
    ],
    links=[
        LinkModel(source='user', target='web', protocol='HTTPS', latency_ms=10),
        LinkModel(source='web', target='api', protocol='HTTPS', latency_ms=6),
        LinkModel(source='api', target='checkout', protocol='gRPC', latency_ms=4),
        LinkModel(source='checkout', target='payments', label='authorize', protocol='HTTPS', latency_ms=20, loss_rate=0.02),
        LinkModel(source='checkout', target='orders', label='persist', protocol='SQL', latency_ms=8),
        LinkModel(source='checkout', target='queue', mode='flow', label='publish', protocol='Kafka', latency_ms=5),
    ],
)

scenario = TrafficScenario(
    name='holiday-peak',
    entry_requests_per_second={'web': 52},
    duration_steps=30,
    step_seconds=300,
    diurnal_amplitude=0.35,
    spike_step=18,
    spike_multiplier=2.1,
)

commerce

## Map Notebook Models to a Properly Notated Diagram

In [ ]:
def to_silco_diagram(model: ArchitectureModel):
    d = diagram(model.title, direction=model.direction)
    for group_id, label in model.groups.items():
        d.group(group_id, label=label)
    for service in model.services:
        metadata = {}
        if service.technology:
            metadata['technology'] = service.technology
        if service.line is not None:
            metadata['line'] = service.line
        d.node(
            service.id,
            service.label,
            kind=service.kind,
            group=service.group,
            description=service.description,
            **metadata,
        )
    for link in model.links:
        if link.mode == 'flow':
            d.flow(link.source, link.target, link.label, protocol=link.protocol, calls_per_request=link.calls_per_request)
        else:
            d.connect(link.source, link.target, link.label, protocol=link.protocol, calls_per_request=link.calls_per_request)
    return d


commerce_diagram = to_silco_diagram(commerce)
try:
    display(SVG(commerce_diagram.to_svg(style='modern')))
except RuntimeError as exc:
    print('SVG backend unavailable in this kernel:', exc)
print(commerce_diagram.to_mermaid())

## Advanced Flow Simulation

In [ ]:
def topology(model: ArchitectureModel):
    indegree = {service.id: 0 for service in model.services}
    outgoing = defaultdict(list)
    for link in model.links:
        outgoing[link.source].append(link)
        indegree[link.target] += 1
    queue = deque([service_id for service_id, count in indegree.items() if count == 0])
    order = []
    while queue:
        current = queue.popleft()
        order.append(current)
        for link in outgoing[current]:
            indegree[link.target] -= 1
            if indegree[link.target] == 0:
                queue.append(link.target)
    if len(order) != len(model.services):
        raise ValueError('Simulation requires an acyclic graph.')
    return order, outgoing


def demand_multiplier(step: int, total_steps: int, scenario: TrafficScenario) -> float:
    diurnal = 1.0 + scenario.diurnal_amplitude * sin(2 * pi * step / max(total_steps, 1))
    spike = scenario.spike_multiplier if scenario.spike_step == step else 1.0
    return max(0.1, diurnal * spike)


def simulate_advanced(model: ArchitectureModel, scenario: TrafficScenario):
    order, outgoing = topology(model)
    services = {service.id: service for service in model.services}
    async_backlog = defaultdict(float)
    timeline = []
    per_service_totals = defaultdict(lambda: defaultdict(float))
    per_edge_totals = defaultdict(float)

    for step in range(scenario.duration_steps):
        multiplier = demand_multiplier(step, scenario.duration_steps, scenario)
        incoming = defaultdict(float)
        edge_flows = {}
        service_frame = {}

        for entry_id, base_rps in scenario.entry_requests_per_second.items():
            incoming[entry_id] += base_rps * multiplier
        for service_id, backlog in list(async_backlog.items()):
            incoming[service_id] += backlog

        next_backlog = defaultdict(float)

        for service_id in order:
            service = services[service_id]
            requested_rps = incoming[service_id]
            capacity = service.replicas * service.capacity_rps_per_replica
            served_rps = min(requested_rps, capacity)
            saturated_rps = max(0.0, requested_rps - capacity)
            failures_rps = served_rps * service.failure_rate
            retries_rps = failures_rps * service.retry_multiplier
            successful_rps = max(0.0, served_rps - failures_rps)
            utilization = served_rps / capacity if capacity else 0.0
            latency_ms = service.base_latency_ms * (1.0 + utilization ** 2.2)
            required_replicas = max(1, ceil(requested_rps / service.capacity_rps_per_replica)) if requested_rps else 1
            recommended_replicas = min(service.max_replicas, max(service.replicas, required_replicas))
            step_requests = successful_rps * scenario.step_seconds
            step_cost = service.monthly_base_cost / (30 * 24 * 3600 / scenario.step_seconds) + service.cost_per_request * step_requests

            service_frame[service_id] = {
                'label': service.label,
                'requested_rps': requested_rps,
                'served_rps': served_rps,
                'successful_rps': successful_rps,
                'failures_rps': failures_rps,
                'saturated_rps': saturated_rps,
                'retries_rps': retries_rps,
                'latency_ms': latency_ms,
                'utilization': utilization,
                'replicas': service.replicas,
                'recommended_replicas': recommended_replicas,
                'step_cost': step_cost,
            }

            per_service_totals[service_id]['requests'] += step_requests
            per_service_totals[service_id]['cost'] += step_cost
            per_service_totals[service_id]['failures'] += failures_rps * scenario.step_seconds
            per_service_totals[service_id]['saturated'] += saturated_rps * scenario.step_seconds

            forwarded_budget = successful_rps + retries_rps
            for link in outgoing[service_id]:
                flow_rps = forwarded_budget * link.calls_per_request
                flow_rps = flow_rps * (1.0 - link.loss_rate)
                edge_key = (link.source, link.target)
                edge_flows[edge_key] = {
                    'rps': flow_rps,
                    'mode': link.mode,
                    'protocol': link.protocol or '',
                    'label': link.label or '',
                    'latency_ms': link.latency_ms,
                }
                per_edge_totals[edge_key] += flow_rps * scenario.step_seconds
                if link.mode == 'flow':
                    decay = 1.0 - services[link.target].queue_decay
                    next_backlog[link.target] += flow_rps * decay
                else:
                    incoming[link.target] += flow_rps

        async_backlog = next_backlog
        timeline.append({
            'step': step,
            'multiplier': multiplier,
            'services': service_frame,
            'edges': edge_flows,
        })

    service_summary = []
    for service in model.services:
        totals = per_service_totals[service.id]
        peak = max(frame['services'][service.id]['requested_rps'] for frame in timeline)
        p95_latency = sorted(frame['services'][service.id]['latency_ms'] for frame in timeline)[int(0.95 * (len(timeline) - 1))]
        service_summary.append({
            'service': service.label,
            'peak_requested_rps': round(peak, 2),
            'total_requests': round(totals['requests'], 0),
            'failed_requests': round(totals['failures'], 0),
            'saturated_requests': round(totals['saturated'], 0),
            'p95_latency_ms': round(p95_latency, 2),
            'total_cost': round(totals['cost'], 2),
        })

    edge_summary = []
    for link in model.links:
        edge_key = (link.source, link.target)
        peak_edge_rps = max(frame['edges'].get(edge_key, {}).get('rps', 0.0) for frame in timeline)
        edge_summary.append({
            'source': link.source,
            'target': link.target,
            'mode': link.mode,
            'protocol': link.protocol,
            'peak_rps': round(peak_edge_rps, 2),
            'total_messages': round(per_edge_totals[edge_key], 0),
        })

    return {
        'timeline': timeline,
        'service_summary': service_summary,
        'edge_summary': edge_summary,
        'scenario': scenario,
        'model': model,
    }

## Run the Simulation and Show Explicit Flows

In [ ]:
sim = simulate_advanced(commerce, scenario)
service_summary = sorted(sim['service_summary'], key=lambda row: row['total_cost'], reverse=True)
edge_summary = sorted(sim['edge_summary'], key=lambda row: row['total_messages'], reverse=True)

print('Scenario:', scenario.name)
print('Steps:', scenario.duration_steps)
print('Total estimated cost:', round(sum(row['total_cost'] for row in service_summary), 2))
service_summary

In [ ]:
print('Per-link flow summary')
edge_summary

## Animate Flows On The Diagram

The animation below keeps the actual architecture diagram on screen and animates flow directly on top of it.

In [ ]:
def _parse_points(points: str):
    values = [float(value) for pair in points.split() for value in pair.split(',')]
    xs = values[0::2]
    ys = values[1::2]
    return min(xs), min(ys), max(xs), max(ys)


def _parse_path_bbox(path_d: str):
    values = [float(value) for value in re.findall(r'-?\d+(?:\.\d+)?', path_d)]
    xs = values[0::2]
    ys = values[1::2]
    return min(xs), min(ys), max(xs), max(ys)


def _extract_svg_geometry(svg: str):
    root = ET.fromstring(svg)
    ns = {'svg': 'http://www.w3.org/2000/svg'}
    view_box = [float(value) for value in root.attrib['viewBox'].split()]
    _, _, svg_width, svg_height = view_box
    nodes = {}
    edges = {}

    for group in root.findall('.//svg:g[@class="node"]', ns):
        title = group.find('svg:title', ns)
        if title is None or not title.text:
            continue
        node_id = title.text.strip()
        bounds = None
        for child in list(group):
            tag = child.tag.split('}')[-1]
            if tag == 'polygon' and 'points' in child.attrib:
                bounds = _parse_points(child.attrib['points'])
                break
            if tag == 'image':
                x = float(child.attrib.get('x', 0.0))
                y = float(child.attrib.get('y', 0.0))
                width = float(child.attrib.get('width', '0').replace('px', ''))
                height = float(child.attrib.get('height', '0').replace('px', ''))
                bounds = (x, y, x + width, y + height)
                break
            if tag == 'path' and 'd' in child.attrib:
                bounds = _parse_path_bbox(child.attrib['d'])
                break
        if bounds is None:
            continue
        x1, y1, x2, y2 = bounds
        nodes[node_id] = ((x1 + x2) / 2 / svg_width, (y1 + y2) / 2 / svg_height)

    for group in root.findall('.//svg:g[@class="edge"]', ns):
        title = group.find('svg:title', ns)
        if title is None or not title.text or '->' not in title.text:
            continue
        source, target = [part.strip() for part in title.text.split('->', 1)]
        path = group.find('svg:path', ns)
        if path is None or 'd' not in path.attrib:
            continue
        coords = [float(value) for value in re.findall(r'-?\d+(?:\.\d+)?', path.attrib['d'])]
        if len(coords) < 4:
            continue
        start = (coords[0] / svg_width, coords[1] / svg_height)
        end = (coords[-2] / svg_width, coords[-1] / svg_height)
        edges[(source, target)] = (start, end)

    return nodes, edges


def render_diagram_panel_image(model: ArchitectureModel, style: str = 'modern'):
    rendered = to_silco_diagram(model)
    svg = rendered.to_svg(style=style)
    node_positions, edge_positions = _extract_svg_geometry(svg)
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        svg_path = tmpdir / 'diagram.svg'
        png_path = tmpdir / 'diagram.png'
        svg_path.write_text(svg, encoding='utf-8')
        subprocess.run(['rsvg-convert', '-o', str(png_path), str(svg_path)], check=True)
        image = PILImage.open(png_path).convert('RGBA').copy()
        original_width, original_height = image.size
        background = PILImage.new('RGBA', image.size, (255, 255, 255, 255))
        diff = ImageChops.difference(image, background)
        bbox = diff.getbbox()
        if bbox is None:
            return image, (0.0, 0.0, 1.0, 1.0)
        pad = 24
        left = max(0, bbox[0] - pad)
        top = max(0, bbox[1] - pad)
        right = min(original_width, bbox[2] + pad)
        bottom = min(original_height, bbox[3] + pad)
        cropped = image.crop((left, top, right, bottom))
        bounds = (
            left / original_width,
            top / original_height,
            right / original_width,
            bottom / original_height,
        )
        return cropped, bounds, node_positions, edge_positions


def _transform_positions(positions, bounds):
    left, top, right, bottom = bounds
    width = max(right - left, 1e-6)
    height = max(bottom - top, 1e-6)
    transformed = {}
    for key, (x, y) in positions.items():
        transformed[key] = ((x - left) / width, (y - top) / height)
    return transformed


def _transform_edges(edges, bounds):
    left, top, right, bottom = bounds
    width = max(right - left, 1e-6)
    height = max(bottom - top, 1e-6)
    transformed = {}
    for key, ((x1, y1), (x2, y2)) in edges.items():
        transformed[key] = (((x1 - left) / width, (y1 - top) / height), ((x2 - left) / width, (y2 - top) / height))
    return transformed


def _draw_flow_overlay(ax, model: ArchitectureModel, frame, positions, edge_segments, max_edge_rps, phase: float):
    hot_edges = sorted(frame['edges'].items(), key=lambda item: item[1]['rps'], reverse=True)[:3]
    for service in model.services:
        if service.id not in positions:
            continue
        x, y = positions[service.id]
        metrics = frame['services'][service.id]
        util = metrics['utilization']
        ring_color = '#dc2626' if util > 0.95 else '#f59e0b' if util > 0.75 else '#10b981'
        ring_size = 250 + 350 * min(util, 1.0)
        ax.scatter([x], [y], s=ring_size, facecolors='none', edgecolors=ring_color, linewidths=2.0, alpha=0.9)

    for link in model.links:
        edge_key = (link.source, link.target)
        if edge_key not in edge_segments:
            continue
        (x1, y1), (x2, y2) = edge_segments[edge_key]
        edge_info = frame['edges'].get(edge_key, {'rps': 0.0, 'mode': link.mode, 'protocol': link.protocol or ''})
        rps = edge_info['rps']
        intensity = rps / max_edge_rps if max_edge_rps else 0.0
        color = '#d97706' if edge_info['mode'] == 'flow' else '#2563eb'
        linestyle = ':' if edge_info['mode'] == 'flow' else '--'
        ax.plot([x1, x2], [y1, y2], linestyle=linestyle, linewidth=1.5 + 3.5 * intensity, color=color, alpha=0.25 + 0.65 * intensity)
        particle_count = max(1, min(6, int(round(1 + intensity * 5)))) if rps > 0 else 0
        for particle_index in range(particle_count):
            t = (phase + particle_index / max(particle_count, 1)) % 1.0
            px = x1 + (x2 - x1) * t
            py = y1 + (y2 - y1) * t
            ax.scatter([px], [py], s=35 + 50 * intensity, color=color, alpha=0.95, edgecolors='white', linewidths=0.6)
        if rps > 0.05:
            mx = x1 + (x2 - x1) * 0.52
            my = y1 + (y2 - y1) * 0.52
            ax.text(mx, my - 0.025, f"{rps:.1f} rps", fontsize=7, color='#111827', ha='center', va='center', bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.7, 'pad': 0.15})

    hot_text = '\n'.join(f"{src}->{dst}: {info['rps']:.1f} rps" for (src, dst), info in hot_edges)
    if hot_text:
        ax.text(0.02, 0.02, hot_text, transform=ax.transAxes, fontsize=8, color='#111827', va='bottom', ha='left', bbox={'facecolor': 'white', 'edgecolor': '#cbd5e1', 'alpha': 0.92, 'boxstyle': 'round,pad=0.3'})


def animate_flow_diagram(model: ArchitectureModel, sim_result):
    timeline = sim_result['timeline']
    edge_keys = [(link.source, link.target) for link in model.links]
    max_edge_rps = max(frame['edges'].get(edge_key, {}).get('rps', 0.0) for frame in timeline for edge_key in edge_keys) or 1.0
    diagram_image, bounds, node_positions, edge_positions = render_diagram_panel_image(model)
    positions = _transform_positions(node_positions, bounds)
    edge_segments = _transform_edges(edge_positions, bounds)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))

    def render(frame_index: int):
        ax.clear()
        frame = timeline[frame_index]
        ax.set_facecolor('white')
        ax.imshow(diagram_image, extent=(0, 1, 1, 0))
        ax.set_title(f"{model.title} | step {frame_index + 1}/{len(timeline)} | multiplier {frame['multiplier']:.2f}", fontsize=12)
        ax.axis('off')
        phase = (frame_index * 0.18) % 1.0
        _draw_flow_overlay(ax, model, frame, positions, edge_segments, max_edge_rps, phase)
        total_ingress = sum(frame['services'][service_id]['requested_rps'] for service_id in sim_result['scenario'].entry_requests_per_second)
        ax.text(0.98, 0.02, f"Ingress {total_ingress:.1f} rps", transform=ax.transAxes, fontsize=8, color='#111827', va='bottom', ha='right', bbox={'facecolor': 'white', 'edgecolor': '#cbd5e1', 'alpha': 0.92, 'boxstyle': 'round,pad=0.3'})
        fig.tight_layout()

    animation = FuncAnimation(fig, render, frames=len(timeline), interval=450, repeat=True)
    plt.close(fig)
    return animation


def save_flow_diagram_gif(model: ArchitectureModel, sim_result, path='commerce-flow-diagram.gif'):
    timeline = sim_result['timeline']
    edge_keys = [(link.source, link.target) for link in model.links]
    max_edge_rps = max(frame['edges'].get(edge_key, {}).get('rps', 0.0) for frame in timeline for edge_key in edge_keys) or 1.0
    diagram_image, bounds, node_positions, edge_positions = render_diagram_panel_image(model)
    positions = _transform_positions(node_positions, bounds)
    edge_segments = _transform_edges(edge_positions, bounds)
    fig, ax = plt.subplots(figsize=(11.5, 4.6))

    def render(frame_index: int):
        ax.clear()
        frame = timeline[frame_index]
        ax.set_facecolor('white')
        ax.imshow(diagram_image, extent=(0, 1, 1, 0))
        ax.set_title(f"{model.title} | step {frame_index + 1}/{len(timeline)} | multiplier {frame['multiplier']:.2f}", fontsize=12)
        ax.axis('off')
        phase = (frame_index * 0.18) % 1.0
        _draw_flow_overlay(ax, model, frame, positions, edge_segments, max_edge_rps, phase)
        total_ingress = sum(frame['services'][service_id]['requested_rps'] for service_id in sim_result['scenario'].entry_requests_per_second)
        ax.text(0.98, 0.02, f"Ingress {total_ingress:.1f} rps", transform=ax.transAxes, fontsize=8, color='#111827', va='bottom', ha='right', bbox={'facecolor': 'white', 'edgecolor': '#cbd5e1', 'alpha': 0.92, 'boxstyle': 'round,pad=0.3'})
        fig.tight_layout()

    animation = FuncAnimation(fig, render, frames=len(timeline), interval=450, repeat=False)
    animation.save(path, writer=PillowWriter(fps=2))
    plt.close(fig)
    return Path(path)


In [ ]:
flow_diagram_animation = animate_flow_diagram(commerce, sim)
HTML(flow_diagram_animation.to_jshtml())

In [ ]:
gif_path = save_flow_diagram_gif(commerce, sim)
print('Wrote', gif_path.resolve())
display(IPyImage(filename=str(gif_path)))